In [3]:
from langchain_community.document_loaders import CSVLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.question_answering import load_qa_chain

import os

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")  # Set key via environment variable
os.environ["OPENAI_API_BASE"] = "https://openai.vocareum.com/v1"

loader = CSVLoader(file_path='./tv-reviews.csv')
docs = loader.load()

# print(docs)
model_name = 'gpt-3.5-turbo'
llm = ChatOpenAI(model=model_name, temperature=0, max_tokens=2000)

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
split_docs = splitter.split_documents(docs)

embeddings = OpenAIEmbeddings()

db = Chroma.from_documents(split_docs, embeddings)
query = """
    Based on the reviews in the context, tell me what people liked about the picture quality.
    Make sure you do not paraphrase the reviews, and only use the information provided in the reviews.
    """

use_chain_helper = False
if use_chain_helper:
    rag = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=db.as_retriever())
    print(rag.run(query))
else:
    similar_docs = db.similarity_search(query, k=5)
    prompt = PromptTemplate(
        template="{query}\nContext: {context}",
        input_variables=["query", "context"],
    )
    chain = load_qa_chain(llm, prompt=prompt, chain_type="stuff")
    print(chain.run(input_documents=similar_docs, query=query))

C:\Users\dilip\AppData\Local\Temp\ipykernel_27256\3878751207.py:42: LangChainDeprecationWarning: This class is deprecated. See the following migration guides for replacements based on `chain_type`:
stuff: https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain
map_reduce: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain
refine: https://python.langchain.com/docs/versions/migrating_chains/refine_chain
map_rerank: https://python.langchain.com/docs/versions/migrating_chains/map_rerank_docs_chain

See also guides on retrieval and question-answering here: https://python.langchain.com/docs/how_to/#qa-with-rag
  chain = load_qa_chain(llm, prompt=prompt, chain_type="stuff")
C:\Users\dilip\AppData\Local\Temp\ipykernel_27256\3878751207.py:43: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  print(chain.run(input_documents=similar_docs, query=query))


People liked the vibrant colors and crystal clear images of the picture quality. They found the details to be sharp and lifelike, creating a stunning and immersive viewing experience.
